# 📈 Regresión lineal con scikit-learn
**Curso de IA · Bloque de Machine Learning · Sesión 2**

---

## ¿Qué es una regresión?

Una **regresión** es un tipo de problema de Machine Learning donde queremos **predecir un número**:
el precio de un piso, la temperatura de mañana, las ventas del próximo mes...

La **regresión lineal** es el modelo más sencillo y uno de los más usados. Su idea es muy intuitiva:

> Dibuja la **recta** que mejor se ajusta a la nube de puntos y úsala para predecir.

**Ejemplo:** cuanto más grande es un piso, más caro suele ser. Si dibujamos m² en el eje horizontal y precio en el vertical, los puntos suben de izquierda a derecha. La regresión lineal encuentra la recta que pasa "lo más cerca posible" de todos los puntos.

La ecuación de esa recta es:

```
precio = b + a × m²
```

- **a** (pendiente o *coeficiente*): cuántos euros sube el precio por cada m² extra.
- **b** (ordenada en el origen o *intercepto*): el precio "de partida" cuando m² = 0 (es un punto matemático, no tiene por qué tener sentido real).

Con **varias variables** (regresión múltiple) es lo mismo, pero con más términos:

```
precio = b + a1 × m² + a2 × habitaciones + a3 × antigüedad + a4 × distancia
```

## ¿Cómo encuentra la mejor recta?

Con el método de **mínimos cuadrados**: prueba rectas y se queda con la que **hace más pequeña la suma de los errores al cuadrado** (la distancia vertical entre cada punto y la recta). No hace falta que los alumnos lo calculen: scikit-learn lo hace por nosotros con `.fit()`.

## Plan de la sesión

| # | Paso | Pregunta que nos hacemos |
|---|------|--------------------------|
| 1 | Cargar y explorar | ¿Qué relación hay entre las variables y el precio? |
| 2 | Separar entrenamiento y prueba | ¿Cómo comprobar que el modelo funciona con datos nuevos? |
| 3 | Preparar (nulos) | ¿Cómo rellenar huecos sin "hacer trampa"? |
| 4 | Modelo base | ¿Qué resultado tendría un modelo "tonto"? |
| 5 | Regresión lineal simple | ¿Cuánto vale cada m² extra? |
| 6 | Regresión lineal múltiple | ¿Mejora si usamos más variables? |
| 7 | Evaluar con métricas y gráficas | ¿Cuánto se equivoca y dónde? |
| 8 | Comparar con árboles y ver sobreajuste | ¿Un modelo más complejo siempre es mejor? |
| 9 | Limitaciones | ¿Qué le pasa a la recta con valores extremos? |
| 10 | Predecir y Pipeline | ¿Cómo lo haríamos en un proyecto real? |

## Cómo usar este cuaderno

- Ejecuta cada celda con **Shift + Enter**, **en orden**.
- Si algo falla: **Entorno de ejecución → Reiniciar y ejecutar todo**.
- Lee el texto antes de ejecutar cada celda. **Predice el resultado** y luego compruébalo.

> 👩‍🏫 **Guía de tiempos para el profesor** (orientativa, total ≈ 7 h con ejercicios)
>
> | Sección | Minutos |
> |---|---|
> | 1-2. Librerías y creación del CSV | 25 |
> | 3. Cargar y explorar (gráficas y correlación) | 40 |
> | 4. Separar entrenamiento y prueba | 25 |
> | 5. Rellenar nulos sin "trampa" | 20 |
> | 6. Modelo base | 20 |
> | 7. Regresión lineal simple | 50 |
> | 8. Regresión lineal múltiple | 50 |
> | 9. Evaluar: métricas y gráficas | 45 |
> | 10. Sobreajuste: comparar con árboles | 40 |
> | 11. Limitaciones: valores extremos | 30 |
> | 12. Predecir con datos nuevos y Pipeline | 25 |
> | 13-14. Resumen, ejercicios y reto | 50 |
>
> Este cuaderno da por conocida la sesión de limpieza (nulos, mediana, escalado). Aquí **aplicamos correctamente** lo que se dejó anotado: calcular las medianas **solo con el conjunto de entrenamiento**.

## 1. Importar las librerías

Además de las de siempre (pandas, numpy, matplotlib), importaremos de **scikit-learn** lo necesario a medida que lo vayamos usando, para que se vea de dónde viene cada pieza.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (7, 4)   # tamaño por defecto de los gráficos

import sklearn
print("Versión de scikit-learn:", sklearn.__version__)

## 2. Crear nuestro CSV de viviendas

Para no depender de ficheros externos, la celda siguiente **fabrica** un CSV con 300 viviendas y lo guarda como `viviendas.csv`.

| Columna | Qué es |
|---|---|
| `m2` | Metros cuadrados |
| `habitaciones` | Número de habitaciones |
| `antiguedad` | Años del edificio |
| `distancia_centro` | Distancia al centro en km |
| `precio` | Precio en euros ← **lo que queremos predecir** |

Algo importante de vocabulario:

- Las columnas que usamos para predecir (m², habitaciones...) se llaman **variables independientes**, **características** o **features** y se suelen llamar **X**.
- La columna que queremos predecir (precio) se llama **variable objetivo** o **target** y se suele llamar **y**.

El CSV tiene algunos **nulos** a propósito (para repasar la sesión de limpieza). Los precios los hemos generado con una fórmula + un poco de **ruido aleatorio** (porque en la vida real nada es perfecto). Al final del cuaderno veremos si el modelo "descubre" esa fórmula.

> 💡 No hace falta entender el código de esta celda: es la "fábrica" de datos.

In [ ]:
rng = np.random.default_rng(7)   # semilla: todos obtenemos los mismos datos
n = 300

m2 = rng.uniform(40, 200, n).round(0)
habitaciones = np.clip(np.round(m2 / 40 + rng.normal(0, 0.8, n)), 1, 6).astype(int)
antiguedad = rng.integers(0, 61, n).astype(float)
distancia_centro = rng.uniform(0.5, 25, n).round(1)
ruido = rng.normal(0, 25000, n)

precio = (30000 + 2200 * m2 + 9000 * habitaciones
          - 900 * antiguedad - 3500 * distancia_centro + ruido).round(0)

df_fabrica = pd.DataFrame({
    "m2": m2,
    "habitaciones": habitaciones,
    "antiguedad": antiguedad,
    "distancia_centro": distancia_centro,
    "precio": precio,
})

# Metemos algunos nulos a propósito
df_fabrica.loc[rng.choice(n, 8, replace=False), "antiguedad"] = np.nan
df_fabrica.loc[rng.choice(n, 5, replace=False), "distancia_centro"] = np.nan

df_fabrica.to_csv("viviendas.csv", index=False)
print("✅ Fichero 'viviendas.csv' creado con", len(df_fabrica), "filas")

> 📁 **¿Prefieres un CSV externo?** Sube tu fichero con el icono de la carpeta 📁 (barra izquierda de Colab) y cambia el nombre en `pd.read_csv(...)` del apartado siguiente. Recuerda cambiar también el nombre de la columna objetivo (`"precio"`) donde aparezca. Al final del cuaderno tienes un reto con un dataset real.

## 3. Cargar y explorar los datos

Antes de entrenar nada hay que **entender los datos**: ¿qué rango tienen?, ¿cómo se relacionan con el precio?

In [ ]:
df = pd.read_csv("viviendas.csv")
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
df.info()

👀 Fíjate en `Non-Null Count`: `antiguedad` y `distancia_centro` tienen menos valores que el resto. Son los nulos que tendremos que tratar.

In [ ]:
df.describe().round(1)

### 3.1 ¿Cómo se distribuye el precio?

Un **histograma** divide el rango de precios en "cajitas" y cuenta cuántas viviendas caen en cada una.

In [ ]:
plt.hist(df["precio"], bins=25, color="steelblue", edgecolor="white")
plt.title("Distribución del precio")
plt.xlabel("Precio (€)")
plt.ylabel("Número de viviendas")
plt.show()

### 3.2 ¿Cómo se relaciona cada variable con el precio?

Un **diagrama de dispersión** (scatter) pone un punto por vivienda. Si la nube de puntos tiene forma de línea, la regresión lineal funcionará bien.

In [ ]:
variables = ["m2", "habitaciones", "antiguedad", "distancia_centro"]

fig, ejes = plt.subplots(2, 2, figsize=(11, 8))
for eje, col in zip(ejes.flatten(), variables):
    eje.scatter(df[col], df["precio"], alpha=0.5, s=15)
    eje.set_xlabel(col)
    eje.set_ylabel("precio")
    eje.set_title(f"{col} vs precio")
plt.tight_layout()
plt.show()

👀 **¿Qué observas?**
- `m2`: los puntos **suben** → a más metros, más precio (relación positiva).
- `antiguedad` y `distancia_centro`: los puntos **bajan** → a más años o más lejos, menos precio (relación negativa).
- `habitaciones`: sube, pero se ve más "en escalones" porque solo toma valores enteros.

### 3.3 Correlación: medir la relación con un número

La **correlación** mide con un número entre **-1 y 1** cuánto se mueven juntas dos variables:

| Valor | Significado |
|---|---|
| Cerca de **+1** | Cuando una sube, la otra sube (relación positiva fuerte) |
| Cerca de **-1** | Cuando una sube, la otra baja (relación negativa fuerte) |
| Cerca de **0** | No hay relación lineal |

⚠️ **Correlación no es causalidad:** que dos cosas vayan juntas no significa que una cause la otra.

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)

# Escribimos el valor dentro de cada casilla
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)

plt.colorbar(im)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()

👀 Mira la **fila o columna `precio`**: `m2` tiene la correlación positiva más alta. Fíjate además en que `m2` y `habitaciones` también están muy correlacionadas entre sí (los pisos grandes tienen más habitaciones). Esto tendrá consecuencias más adelante.

## 4. Separar entrenamiento y prueba (¡el paso más importante!)

**Analogía del examen:** si un profesor da a sus alumnos el examen y las respuestas para estudiar, y luego les pone ese mismo examen, todos sacarán un 10... pero eso no demuestra que hayan aprendido.

Con los modelos pasa igual. Dividimos los datos en dos:

| Conjunto | Para qué sirve | Analogía |
|---|---|---|
| **Entrenamiento (train)** | El modelo aprende de él | Los apuntes y ejercicios |
| **Prueba (test)** | Comprobamos qué tal funciona con datos que **nunca ha visto** | El examen |

Se suele reservar entre un **20 % y un 30 %** para prueba.

Primero separamos **X** (características) e **y** (objetivo):

In [ ]:
X = df.drop(columns="precio")   # todas las columnas menos el precio
y = df["precio"]                # solo el precio

print("X tiene", X.shape[1], "columnas:", list(X.columns))
print("y es la columna:", y.name)

Ahora los separamos con `train_test_split`:

- `test_size=0.2` → el 20 % para prueba.
- `random_state=42` → fija el "sorteo" para que **todos obtengan la misma división** (si no, cada ejecución daría una distinta). El número 42 es una tradición, vale cualquiera.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Entrenamiento:", X_train.shape[0], "viviendas")
print("Prueba       :", X_test.shape[0], "viviendas")

> 🔒 **Regla de oro desde ahora:** el conjunto de prueba **no se toca** hasta el momento de evaluar. Todo lo que el modelo "aprenda" (incluso rellenar nulos) debe salir **solo** del entrenamiento.

## 5. Rellenar los nulos sin "hacer trampa"

En la sesión de limpieza rellenamos nulos con la mediana de **todo** el dataset. Pero eso hace que información del conjunto de prueba "se cuele" en el entrenamiento. A esto se le llama **fuga de datos** (*data leakage*) y hace que los resultados parezcan mejores de lo que son.

**La forma correcta** con scikit-learn usa un objeto `SimpleImputer` con dos pasos:

| Método | Qué hace | Se aplica a |
|---|---|---|
| `.fit()` | **Aprende** la mediana de cada columna | **Solo entrenamiento** |
| `.transform()` | **Rellena** los nulos con lo aprendido | Entrenamiento **y** prueba |

(El método `.fit_transform()` hace los dos pasos seguidos; solo se usa con el entrenamiento.)

Veamos primero cuántos nulos hay en cada conjunto:

In [ ]:
print("Nulos en entrenamiento:")
print(X_train.isna().sum())
print("\nNulos en prueba:")
print(X_test.isna().sum())

In [ ]:
from sklearn.impute import SimpleImputer

imputador = SimpleImputer(strategy="median")

# Aprende las medianas SOLO con entrenamiento y rellena
X_train_imp = pd.DataFrame(
    imputador.fit_transform(X_train), columns=X.columns, index=X_train.index
)

# Con prueba solo rellena (transform), con las medianas aprendidas en entrenamiento
X_test_imp = pd.DataFrame(
    imputador.transform(X_test), columns=X.columns, index=X_test.index
)

print("Medianas aprendidas en entrenamiento:")
print(pd.Series(imputador.statistics_, index=X.columns).round(1))

print("\nNulos que quedan -> train:", int(X_train_imp.isna().sum().sum()),
      "| test:", int(X_test_imp.isna().sum().sum()))

## 6. Modelo base: el modelo "tonto"

Antes de usar un modelo inteligente, necesitamos saber **qué resultado tendría uno que no lo es**. Si nuestro modelo no lo supera, no sirve para nada.

El `DummyRegressor` con `strategy="mean"` **predice siempre el precio medio**, sin mirar ninguna característica.

### Cómo se usa cualquier modelo de scikit-learn (¡siempre igual!)

```python
modelo = TipoDeModelo()          # 1. Crear
modelo.fit(X_train, y_train)     # 2. Entrenar (aprender)
predicciones = modelo.predict(X_test)   # 3. Predecir
```

### Las tres métricas de regresión

Antes de medir, repasemos qué significan:

| Métrica | Qué mide | Cómo leerla |
|---|---|---|
| **MAE** (error absoluto medio) | El error **medio** en euros | "De media me equivoco en X €" |
| **RMSE** (raíz del error cuadrático medio) | Como el MAE, pero **castiga más los errores grandes** | Siempre ≥ MAE. Si es mucho mayor, hay fallos puntuales grandes |
| **R²** | Qué fracción de la variación del precio explica el modelo | 1 = perfecto · 0 = no mejora a predecir la media · negativo = peor que la media |

Definimos una función para calcularlas y otra para ir **guardando los resultados de cada modelo** en una tabla comparativa:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return mae, rmse, r2

resultados = []   # aquí iremos guardando cada modelo

def registrar(nombre, modelo, Xtr, Xte):
    _, _, r2_tr = metricas(y_train, modelo.predict(Xtr))
    mae_te, rmse_te, r2_te = metricas(y_test, modelo.predict(Xte))
    resultados.append({
        "modelo": nombre,
        "MAE (test)": round(mae_te),
        "RMSE (test)": round(rmse_te),
        "R² (train)": round(r2_tr, 3),
        "R² (test)": round(r2_te, 3),
    })

In [ ]:
from sklearn.dummy import DummyRegressor

modelo_base = DummyRegressor(strategy="mean")
modelo_base.fit(X_train_imp, y_train)

registrar("Base (siempre la media)", modelo_base, X_train_imp, X_test_imp)
pd.DataFrame(resultados)

👀 **Interpretación:**
- El **R² ≈ 0** es lo esperado: un modelo que siempre predice la media "explica" cero variación.
- El **MAE** es el error de partida: **este es el número que hay que batir**.

## 7. Regresión lineal simple (una sola variable)

Empezamos con lo más sencillo: predecir el precio **solo con los m²**. Ajustaremos la recta `precio = b + a × m²`.

Seleccionamos la columna con **doble corchete** `[["m2"]]` para que siga siendo una tabla (scikit-learn espera X en 2 dimensiones).

In [ ]:
from sklearn.linear_model import LinearRegression

X_train_m2 = X_train_imp[["m2"]]
X_test_m2 = X_test_imp[["m2"]]

modelo_simple = LinearRegression()
modelo_simple.fit(X_train_m2, y_train)       # aquí "aprende" la mejor recta

a = modelo_simple.coef_[0]       # pendiente
b = modelo_simple.intercept_     # ordenada en el origen

print(f"Ecuación aprendida:  precio = {b:,.0f} + {a:,.0f} × m²")

### Cómo interpretar el resultado

- **Coeficiente (a):** por cada m² adicional, el precio sube de media en esos euros (manteniendo todo lo demás igual).
- **Intercepto (b):** el valor de la recta cuando m² = 0. Matemáticamente necesario, pero no hay que interpretarlo literalmente (no existen pisos de 0 m²).

### Dibujemos la recta

Generamos valores de m² de menor a mayor y pedimos al modelo su predicción: eso dibuja la recta.

In [ ]:
xs = np.linspace(X_train_m2["m2"].min(), X_train_m2["m2"].max(), 100)
ys = modelo_simple.predict(pd.DataFrame({"m2": xs}))

plt.scatter(X_train_m2["m2"], y_train, alpha=0.5, s=15, label="Viviendas (entrenamiento)")
plt.plot(xs, ys, color="red", linewidth=2, label="Recta de regresión")
plt.xlabel("m²")
plt.ylabel("Precio (€)")
plt.title("Regresión lineal simple: m² → precio")
plt.legend()
plt.show()

### Los errores (residuos)

La diferencia entre el precio **real** y el **predicho** se llama **residuo**. La recta de mínimos cuadrados es la que hace mínima la suma de los cuadrados de esas distancias verticales.

Lo vemos con 25 viviendas de la prueba (las líneas grises son los errores):

In [ ]:
muestra = X_test_m2.iloc[:25]
y_muestra = y_test.iloc[:25]
pred_muestra = modelo_simple.predict(muestra)

plt.scatter(muestra["m2"], y_muestra, color="steelblue", label="Precio real")
plt.plot(xs, ys, color="red", linewidth=2, label="Recta")
plt.vlines(muestra["m2"], y_muestra, pred_muestra, color="gray", alpha=0.7, label="Error (residuo)")
plt.xlabel("m²")
plt.ylabel("Precio (€)")
plt.title("Errores del modelo en 25 viviendas de prueba")
plt.legend()
plt.show()

### Medir el modelo simple

Evaluamos con el conjunto de **prueba** (datos que no ha visto):

In [ ]:
registrar("Lineal simple (m²)", modelo_simple, X_train_m2, X_test_m2)
pd.DataFrame(resultados)

👀 **Compara con el modelo base:**
- El **MAE baja mucho**: ya no nos equivocamos tanto en euros.
- El **R²** deja de ser 0: con solo los m² explicamos buena parte de la variación del precio.

## 8. Regresión lineal múltiple (varias variables)

Ahora usamos **todas** las características a la vez. El código es casi idéntico: solo cambia el X que le pasamos.

```
precio = b + a1 × m² + a2 × habitaciones + a3 × antigüedad + a4 × distancia
```

In [ ]:
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train_imp, y_train)

coeficientes = pd.Series(modelo_multiple.coef_, index=X_train_imp.columns)

print(f"Intercepto: {modelo_multiple.intercept_:,.0f}\n")
print("Coeficientes (€ por cada unidad de la variable):")
print(coeficientes.round(0))

### Cómo leer los coeficientes

Cada coeficiente dice cuánto cambia el precio **al aumentar esa variable en 1 unidad, manteniendo las demás fijas**:

- `m2` positivo → cada m² extra suma euros.
- `antiguedad` negativo → cada año más de antigüedad **resta** euros.
- `distancia_centro` negativo → cada km más lejos del centro **resta** euros.

> 🔍 **¿El modelo ha descubierto la fórmula?** Los datos se fabricaron con estos valores "reales": **m² = +2.200 · habitaciones = +9.000 · antigüedad = −900 · distancia = −3.500**. Compáralos con los que ha aprendido el modelo.
>
> Si `m2`, `antiguedad` y `distancia_centro` salen muy parecidos pero `habitaciones` se aleja más, **no es un fallo**: `m2` y `habitaciones` están muy relacionadas entre sí (se vio en la correlación), y al modelo le cuesta repartir el mérito entre las dos. A esto se le llama **multicolinealidad**: los coeficientes se vuelven menos fiables aunque las predicciones sigan siendo buenas.

In [ ]:
registrar("Lineal múltiple (4 variables)", modelo_multiple, X_train_imp, X_test_imp)
pd.DataFrame(resultados)

👀 Debería mejorar frente al modelo simple: más información útil → menos error.

### 8.1 ¿Qué variable pesa más? Cuidado con las unidades

Comparar coeficientes directamente **engaña**: el de `m2` está en "€ por m²" y el de `distancia_centro` en "€ por km". Para compararlos de forma justa, **estandarizamos** las variables (media 0, desviación 1; lo vimos en la sesión de limpieza) y reentrenamos. Ahora todos los coeficientes están en la misma escala.

In [ ]:
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler().fit(X_train_imp)     # aprende media y desviación en train
X_train_esc = pd.DataFrame(escalador.transform(X_train_imp), columns=X.columns, index=X_train_imp.index)

modelo_esc = LinearRegression().fit(X_train_esc, y_train)
coef_esc = pd.Series(modelo_esc.coef_, index=X.columns).sort_values()

coef_esc.plot(kind="barh", color=["tomato" if v < 0 else "seagreen" for v in coef_esc])
plt.title("Importancia relativa (coeficientes estandarizados)")
plt.xlabel("Efecto en el precio (€) por cada desviación típica")
plt.axvline(0, color="black", linewidth=0.8)
plt.show()

# Comprobamos que escalar NO cambia las predicciones de una regresión lineal
r2_original = r2_score(y_train, modelo_multiple.predict(X_train_imp))
r2_escalado = r2_score(y_train, modelo_esc.predict(X_train_esc))
print(f"R² (train) sin escalar: {r2_original:.4f} | escalado: {r2_escalado:.4f}")

👀 Dos lecciones:
1. **Ahora sí se pueden comparar**: la barra más larga es la variable con más peso en el precio.
2. **Escalar no cambia las predicciones** de la regresión lineal (el R² es idéntico); solo cambia cómo se interpretan los coeficientes. El escalado es imprescindible en otros modelos (kNN, SVM...), pero aquí solo ayuda a interpretar.

## 9. Evaluar bien: métricas y gráficas

Los números resumen, pero **las gráficas enseñan dónde falla el modelo**. Vamos a dibujar tres para el modelo múltiple sobre el conjunto de prueba:

1. **Real vs. predicho:** si fuera perfecto, todos los puntos estarían sobre la diagonal roja.
2. **Residuos vs. predicho:** los errores deberían repartirse **al azar** alrededor de cero. Si ves una curva o un abanico, el modelo está ignorando algo.
3. **Histograma de residuos:** idealmente con forma de campana centrada en cero.

In [ ]:
y_pred = modelo_multiple.predict(X_test_imp)
residuos = y_test - y_pred

fig, ejes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Real vs predicho
ejes[0].scatter(y_test, y_pred, alpha=0.6, s=18)
limites = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ejes[0].plot(limites, limites, "r--", label="Predicción perfecta")
ejes[0].set_xlabel("Precio real")
ejes[0].set_ylabel("Precio predicho")
ejes[0].set_title("Real vs predicho")
ejes[0].legend()

# 2. Residuos vs predicho
ejes[1].scatter(y_pred, residuos, alpha=0.6, s=18)
ejes[1].axhline(0, color="r", linestyle="--")
ejes[1].set_xlabel("Precio predicho")
ejes[1].set_ylabel("Residuo (real - predicho)")
ejes[1].set_title("Residuos")

# 3. Histograma de residuos
ejes[2].hist(residuos, bins=20, color="steelblue", edgecolor="white")
ejes[2].axvline(0, color="r", linestyle="--")
ejes[2].set_xlabel("Residuo")
ejes[2].set_title("Distribución de residuos")

plt.tight_layout()
plt.show()

👀 **¿Qué buscar?**
- Primera gráfica: puntos pegados a la diagonal = buen modelo.
- Segunda: nube sin forma, a la misma altura a izquierda y derecha. Si se abre como un embudo, el modelo falla más en precios altos.
- Tercera: campana centrada en 0.

### Interpretar el error en euros

Una forma muy útil de explicarlo: el **RMSE** es el "error típico". Usémoslo para dar una **horquilla**:

In [ ]:
mae, rmse, r2 = metricas(y_test, y_pred)

print(f"MAE  = {mae:,.0f} €  -> de media nos equivocamos en esta cantidad")
print(f"RMSE = {rmse:,.0f} €  -> error típico (castiga más los fallos grandes)")
print(f"R²   = {r2:.3f}     -> el modelo explica el {r2*100:.1f} % de la variación del precio")
print(f"\nPrecio medio de las viviendas de prueba: {y_test.mean():,.0f} €")
print(f"El MAE equivale a un {mae / y_test.mean() * 100:.1f} % del precio medio")

## 10. ¿Un modelo más complejo es siempre mejor? Sobreajuste

Comparemos la regresión lineal con dos modelos **más complejos** basados en árboles:

- **Árbol de decisión:** hace preguntas sucesivas ("¿m² > 100? ¿distancia < 5?") y se queda con un valor medio en cada hoja.
- **Random Forest:** muchos árboles que "votan" (promedian).

### Sobreajuste (overfitting)

Es cuando un modelo **memoriza los datos de entrenamiento** en vez de aprender el patrón general. Es como el alumno que se aprende de memoria las respuestas del libro: saca un 10 en los ejercicios conocidos y suspende el examen.

**Cómo detectarlo:** comparar **R² en entrenamiento** con **R² en prueba**.

| R² train | R² test | Diagnóstico |
|---|---|---|
| Alto | Alto (parecido) | ✅ Buen modelo |
| Muy alto | Bastante más bajo | ⚠️ **Sobreajuste** |
| Bajo | Bajo | ⚠️ **Infraajuste**: el modelo es demasiado simple |

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Árbol sin límite de profundidad: puede memorizar
arbol_profundo = DecisionTreeRegressor(random_state=42)
arbol_profundo.fit(X_train_imp, y_train)

# Árbol con profundidad limitada a 4 niveles: obligado a generalizar
arbol_limitado = DecisionTreeRegressor(max_depth=4, random_state=42)
arbol_limitado.fit(X_train_imp, y_train)

# Bosque de 100 árboles
bosque = RandomForestRegressor(n_estimators=100, random_state=42)
bosque.fit(X_train_imp, y_train)

registrar("Árbol sin límite", arbol_profundo, X_train_imp, X_test_imp)
registrar("Árbol (profundidad 4)", arbol_limitado, X_train_imp, X_test_imp)
registrar("Random Forest", bosque, X_train_imp, X_test_imp)

tabla = pd.DataFrame(resultados)
tabla

In [ ]:
# Gráfico: R² en train vs test para cada modelo
posiciones = np.arange(len(tabla))
ancho = 0.38

plt.figure(figsize=(10, 4.5))
plt.bar(posiciones - ancho / 2, tabla["R² (train)"], ancho, label="Entrenamiento", color="seagreen")
plt.bar(posiciones + ancho / 2, tabla["R² (test)"], ancho, label="Prueba", color="darkorange")
plt.xticks(posiciones, tabla["modelo"], rotation=25, ha="right")
plt.ylabel("R²")
plt.title("¿Hay sobreajuste? R² en entrenamiento vs prueba")
plt.legend()
plt.tight_layout()
plt.show()

👀 **Qué esperar y cómo explicarlo:**
- El **árbol sin límite** saca un R² de entrenamiento de 1.0 (perfecto) pero **claramente menor en prueba**: ha memorizado el entrenamiento → **sobreajuste** (la "brecha" entre las dos barras).
- El **árbol con profundidad 4** tiene menos brecha, pero se queda corto en prueba: es demasiado simple. Encontrar el punto medio es el reto de ajustar cualquier modelo.
- El **Random Forest** mejora al árbol suelto, pero sigue teniendo una brecha importante.
- La **regresión lineal múltiple** consigue el **mejor R² en prueba** y casi sin brecha. ¿Por qué? Porque **nuestros datos se generaron con una relación lineal**: el modelo más simple era el adecuado. **Un modelo más complejo no es automáticamente mejor.** Con datos reales (relaciones curvas, interacciones) los árboles y bosques pueden ganar.

## 11. Limitaciones: la regresión lineal y los valores extremos

La regresión lineal intenta minimizar errores **al cuadrado**, así que un valor muy extremo tiene mucha influencia y puede **"tirar" de la recta**.

Experimento: añadimos al entrenamiento **5 pisos de lujo** (pequeños pero carísimos, ~1 millón de euros) y comparamos la recta de antes y la de ahora.

In [ ]:
# Datos de entrenamiento + 5 "pisos de lujo" raros
m2_lujo = pd.DataFrame({"m2": [50, 55, 60, 65, 70]})
precio_lujo = pd.Series([900000, 950000, 1000000, 1050000, 1100000])

X_con_lujo = pd.concat([X_train_m2.reset_index(drop=True), m2_lujo], ignore_index=True)
y_con_lujo = pd.concat([y_train.reset_index(drop=True), precio_lujo], ignore_index=True)

modelo_lujo = LinearRegression().fit(X_con_lujo, y_con_lujo)

print(f"Pendiente SIN pisos de lujo: {modelo_simple.coef_[0]:,.0f} € por m²")
print(f"Pendiente CON pisos de lujo: {modelo_lujo.coef_[0]:,.0f} € por m²")

plt.scatter(X_train_m2["m2"], y_train, alpha=0.4, s=15, label="Viviendas normales")
plt.scatter(m2_lujo["m2"], precio_lujo, color="gold", edgecolor="black", s=60, label="Pisos de lujo (outliers)")
plt.plot(xs, modelo_simple.predict(pd.DataFrame({"m2": xs})), color="red", linewidth=2, label="Recta original")
plt.plot(xs, modelo_lujo.predict(pd.DataFrame({"m2": xs})), color="purple", linewidth=2, linestyle="--", label="Recta con outliers")
plt.xlabel("m²")
plt.ylabel("Precio (€)")
plt.title("Efecto de los valores extremos en la regresión lineal")
plt.legend()
plt.show()

👀 Solo **5 puntos** han cambiado de forma notable la recta: la pendiente ya no representa lo que ocurre con la gran mayoría de viviendas.

**Lecciones:**
1. Hay que **detectar y decidir qué hacer con los outliers** antes de modelar (¡la sesión de limpieza!).
2. La regresión lineal asume una relación **lineal**: si la nube de puntos es curva, la recta no se ajustará bien aunque no haya outliers. En ese caso, hay que transformar variables o usar modelos no lineales (árboles, bosques...).

## 12. Usar el modelo: predecir con datos nuevos

Es para lo que sirve todo esto. Describimos una vivienda **nueva** con una tabla de una fila (con **las mismas columnas y en el mismo orden** que se usaron al entrenar):

In [ ]:
vivienda_nueva = pd.DataFrame({
    "m2": [95],
    "habitaciones": [3],
    "antiguedad": [15],
    "distancia_centro": [4.0],
})

prediccion = modelo_multiple.predict(vivienda_nueva)[0]

print(f"Precio estimado: {prediccion:,.0f} €")
print(f"Horquilla razonable (± 1 RMSE): entre {prediccion - rmse:,.0f} € y {prediccion + rmse:,.0f} €")

Siempre conviene dar la predicción **junto con su margen de error**. Un modelo que dice "este piso vale 247.318 €" parece más preciso de lo que realmente es.

### 12.1 La forma "profesional": Pipeline

Hasta ahora hemos rellenado nulos, escalado y entrenado **paso a paso**, vigilando manualmente que cada cosa se hiciera solo con entrenamiento. Es fácil equivocarse. Un **Pipeline** encadena todos los pasos en un solo objeto:

1. Rellenar nulos (mediana)
2. Escalar (estandarizar)
3. Regresión lineal

Con `.fit()` aprende todo solo con entrenamiento, y con `.predict()` aplica los pasos en orden. **Admite datos con nulos directamente.** Evita la fuga de datos por construcción.

In [ ]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LinearRegression(),
)

pipeline.fit(X_train, y_train)          # ¡con X_train ORIGINAL, con sus nulos!

r2_pipeline = pipeline.score(X_test, y_test)   # .score() en regresión devuelve R²
r2_manual = r2_score(y_test, modelo_multiple.predict(X_test_imp))

print(f"R² en prueba (Pipeline)  : {r2_pipeline:.4f}")
print(f"R² en prueba (paso a paso): {r2_manual:.4f}")

✅ Mismo resultado, pero con **menos código y menos riesgo de error**. En proyectos reales se usan siempre Pipelines. Los usaremos de forma sistemática en las próximas sesiones.

## 13. Resumen

### La receta para cualquier modelo de regresión

| Paso | Código clave |
|---|---|
| 1. Cargar y explorar | `pd.read_csv`, `describe()`, scatter, `corr()` |
| 2. Separar X e y | `X = df.drop(columns="objetivo")`, `y = df["objetivo"]` |
| 3. Separar train/test | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| 4. Preparar (solo con train) | `SimpleImputer`, `StandardScaler` (`fit` en train) |
| 5. Modelo base | `DummyRegressor` |
| 6. Entrenar | `modelo.fit(X_train, y_train)` |
| 7. Predecir | `modelo.predict(X_test)` |
| 8. Medir | `MAE`, `RMSE`, `R²` en **train y test** |
| 9. Graficar | Real vs predicho, residuos |

### Ideas clave

- **Siempre** medir con datos que el modelo no ha visto (**test**).
- **Siempre** compararse con un **modelo base**.
- **R² en train ≫ R² en test** → sobreajuste.
- El modelo más sencillo que funcione bien suele ser el mejor.
- Los **outliers** pueden estropear una regresión lineal.
- Los **coeficientes** son lo que hace especial a la regresión lineal: se pueden **explicar**.

## ✍️ Ejercicios para los alumnos

**Nivel 1 – Repaso**
1. Cambia `test_size` a 0.3 y vuelve a ejecutar el cuaderno desde la sección 4. ¿Cambian mucho las métricas?
2. Cambia `random_state` a otro número. ¿Por qué cambian ligeramente los resultados?
3. ¿Cuánto vale, según el modelo simple, un piso de 120 m²? Calcúlalo con `predict` y también a mano con la ecuación.

**Nivel 2 – Practicar**
4. Entrena un modelo lineal usando **solo `m2` y `distancia_centro`**. ¿Se parece su R² al del modelo con 4 variables? ¿Qué te dice eso sobre `habitaciones` y `antiguedad`?
5. Cambia `strategy="median"` por `"mean"` en el `SimpleImputer`. ¿Se nota la diferencia?
6. Añade un solo piso de lujo (en lugar de 5) en la sección 11. ¿Cuánto cambia la pendiente?

**Nivel 3 – Retos**
7. Prueba `max_depth` = 2, 3, 5, 8 y 12 en el árbol y dibuja cómo evoluciona el R² de train y de test. ¿A partir de qué profundidad empieza el sobreajuste?
8. Usa `Ridge()` (de `sklearn.linear_model`) en lugar de `LinearRegression()`. ¿Qué cambia?
9. Repite todo el proceso con un **dataset real**: *California Housing* (ver abajo).

## 🏠 Reto: un dataset real (California Housing)

scikit-learn trae incluido un dataset real de precios de viviendas en California. La variable objetivo es `MedHouseVal` (precio medio en cientos de miles de dólares). Descarga los datos con la celda siguiente (necesita conexión a internet, que Colab tiene):

In [ ]:
from sklearn.datasets import fetch_california_housing

try:
    california = fetch_california_housing(as_frame=True)
    df_cal = california.frame
    print("Filas y columnas:", df_cal.shape)
    display(df_cal.head())
except Exception as error:
    print("No se pudo descargar el dataset:", error)

**Tu misión:** aplicar la receta de la sección 13 a `df_cal` con `MedHouseVal` como objetivo.

Preguntas para reflexionar:
- ¿Mejora la regresión lineal al modelo base? ¿Cuánto?
- ¿Qué R² obtienes? ¿Es tan alto como con nuestros datos fabricados? **¿Por qué crees que los datos reales dan resultados menos "limpios"?**
- ¿Gana ahora el Random Forest a la regresión lineal?
- Dibuja las gráficas de real vs. predicho y residuos. ¿Qué observas?

---
### 🎯 Próxima sesión
**Clasificación:** en lugar de predecir un número, predeciremos una **categoría** (¿compra o no compra? ¿qué especie de flor es?) con nuevas métricas: accuracy, precision, recall, F1 y matriz de confusión.